In [44]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [45]:
class SelfAttention(nn.Module):
  def __init__(self, d_model, d_k):
    super().__init__()
    self.W_q = nn.Linear(d_model, d_k)
    self.W_k = nn.Linear(d_model, d_k)
    self.W_v = nn.Linear(d_model, d_k)
    self.D_k = d_k

  def forward(self, x):
    print("x: (batch, seq_len, features)", x.shape)

    Q = self.W_q(x)
    K = self.W_k(x)
    V = self.W_v(x)
    print("Shapes: [batch, seq_len, d_model]-> Q:", Q.shape, "K:", K.shape, "V:", V.shape)

    scores = torch.bmm(Q, K.transpose(-2, -1))
    print("Scores(batch, seq_len, seq_len):", scores.shape)

    d_k_root = self.D_k ** 0.5
    scores = scores / d_k_root
    print("scaled scores:", scores.shape)

    attn_weights = F.softmax(scores, dim=-1)
    print("Attention_weights:", attn_weights.shape)

    output = torch.bmm(attn_weights, V)
    print("output:", output.shape)

    return output, attn_weights

In [46]:
batch, seq_len, d_model, d_k = 4, 5, 16, 16

x = torch.randn(batch, seq_len, d_model)
attn = SelfAttention(d_model=d_model, d_k=d_k)
output, weights = attn(x)


x: (batch, seq_len, features) torch.Size([4, 5, 16])
Shapes: [batch, seq_len, d_model]-> Q: torch.Size([4, 5, 16]) K: torch.Size([4, 5, 16]) V: torch.Size([4, 5, 16])
Scores(batch, seq_len, seq_len): torch.Size([4, 5, 5])
scaled scores: torch.Size([4, 5, 5])
Attention_weights: torch.Size([4, 5, 5])
output: torch.Size([4, 5, 16])


In [47]:
print("Output shape :", output.shape)    # batch_size, seq_len, d_k
print("Weights shape:", weights.shape)   # batch_size, seq_len, seq_len
print("Row sums     :", weights.sum(dim=-1))  # ~1.0 each row

Output shape : torch.Size([4, 5, 16])
Weights shape: torch.Size([4, 5, 5])
Row sums     : tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]], grad_fn=<SumBackward1>)


# Plan — 5 chhote chunks:

0.Ek sentence aur chhota vocabulary banao
1.Sentence ko token IDs mein convert karo
2.Embedding layer use karke vectors banao
3.Self-attention chalao (jo tumne already likhi hai)
4.Attention matrix dekho


